In [0]:
base = "/Volumes/cs483eviction/raw/lcbh_raw"
files = [f.name for f in dbutils.fs.ls(base)]

raw = {}
for name in files:
    raw[name] = (spark.read.option("header", True)
                      .option("inferSchema", True)
                      .csv(f"{base}/{name}"))
    print(f"\n=== {name}: {raw[name].count()} rows ===")
    raw[name].printSchema()

In [0]:
for name in ["eviction_data_tract.csv", "census_data_tract.csv", "census_tract_crosswalk_chicago.csv"]:
    print(f"\n=== {name} ===")
    print(raw[name].columns)
    display(raw[name].limit(5))

In [0]:
from pyspark.sql import functions as F

ev = (raw["eviction_data_tract.csv"]
      .withColumn("tract_id", F.lpad(F.col("tract").cast("string"), 11, "0"))
      .withColumnRenamed("filing_year", "year")
      .withColumn("count_missing", F.col("eviction_filings_total").isNull())
      .withColumn("period",
          F.when(F.col("year") < 2020, "pre_covid")
           .when(F.col("year").between(2020, 2022), "covid")
           .otherwise("post_covid"))
      .dropDuplicates(["tract_id", "year"]))

cen = (raw["census_data_tract.csv"]
       .withColumn("tract_id", F.lpad(F.col("tract").cast("string"), 11, "0"))
       .withColumnRenamed("census_year", "acs_period")
       .drop("tract"))

display(cen.select("acs_period").distinct())
display(ev.select("year").distinct().orderBy("year"))

In [0]:
from pyspark.sql import functions as F

cen2 = (cen
        .withColumn("acs_key", F.regexp_extract("acs_period", r"^(\d{4}-\d{4})", 1))
        .drop("acs_period"))

ev2 = ev.withColumn("acs_key",
        F.when(F.col("year") <= 2012, "2006-2010").otherwise("2014-2018"))

print("duplicate census keys:",
      cen2.groupBy("tract_id", "acs_key").count().filter("count > 1").count())

final = (ev2.join(cen2, ["tract_id", "acs_key"], "left")
   .withColumn("eviction_rate_per_100",
       F.when(F.col("housing_units_rental") > 0,
              F.col("eviction_filings_total") / F.col("housing_units_rental") * 100))
   .withColumn("eviction_rate_per_1000",
       F.when(F.col("housing_units_rental") > 0,
              F.col("eviction_filings_total") / F.col("housing_units_rental") * 1000)))

print("eviction rows:", ev.count(), "| final rows:", final.count())
display(final.select("tract_id", "year", "eviction_filings_total",
                     "housing_units_rental", "eviction_rate_per_1000").limit(20))

In [0]:
display(final.select("tract_id", "year", "eviction_filings_rate",
                     "eviction_rate_per_100", "eviction_rate_per_1000")
             .filter("eviction_filings_rate is not null")
             .limit(20))

(final.write.format("delta")
      .mode("overwrite")
      .saveAsTable("cs483eviction.default.lcbh_tract_year"))

print("saved rows:", spark.table("cs483eviction.default.lcbh_tract_year").count())

In [0]:
from pyspark.sql import functions as F

comm = raw["eviction_data_comm_area.csv"]
print(comm.columns)

display(comm.filter(F.col("filing_year") == 2019)
            .orderBy(F.desc("eviction_filings_total"))
            .limit(100))

In [0]:
display(comm.filter(F.col("filing_year") == 2019)
            .orderBy(F.desc("eviction_filings_rate"))
            .select("area_name", "eviction_filings_total", "eviction_filings_rate")
            .limit(5))

In [0]:
df = spark.table("cs483eviction.default.lcbh_tract_year").toPandas()
df.to_csv("/Volumes/cs483eviction/raw/lcbh_raw/lcbh_tract_year.csv", index=False)
print("exported", len(df), "rows")